# RAG from scratch — `rag-ouz`

Build a **Retrieval-Augmented Generation** system over two real medical-coding PDFs,
one cell at a time, running **entirely on your own machine** (no API keys, no internet).

## What problem does RAG solve?

A small language model like Qwen2.5-0.5B knows almost nothing about ICD-10 coding rules,
and if you ask it anyway it will *invent* a confident answer. That is hallucination.

RAG fixes this by changing the question. Instead of asking:

> "What is the rule for coding the underlying cause of death?"

we first **look up** the relevant pages in a real manual, then ask:

> "Here are 4 passages from the WHO ICD-10 manual. Using ONLY these, answer: what is the rule...?"

The model stops being a knowledge source and becomes a **reading-comprehension engine**.
That is a much easier job, which is why a 0.5B model can do it.

## The pipeline you are about to build

```
PDF  ──►  text  ──►  chunks  ──►  embeddings  ──►  vector store
                                                        │
                       question ──► embedding ──────────┤
                                                        ▼
                                                   top-k chunks
                                                        │
                                                        ▼
                                               prompt ──► LLM ──► grounded answer
```

Seven steps, one section each:

| Step | What happens | Library |
|------|--------------|---------|
| 1 | Read text out of the PDFs | `pypdf` |
| 2 | Cut text into overlapping chunks | plain Python |
| 3 | Turn each chunk into a vector | `sentence-transformers` (BAAI/bge-m3) |
| 4 | Store vectors so we can search them | `chromadb` |
| 5 | Find the chunks closest to a question | `chromadb` |
| 6 | Ask a local LLM using those chunks | `transformers` (Qwen2.5-0.5B-Instruct) |
| 7 | Understand where it breaks | your brain |

**Run the cells in order, top to bottom.**

---
# Step 0 — Setup and sanity check

Before anything else: confirm the libraries import and the two models are already on disk.

Both models were downloaded previously and live in the HuggingFace cache
(`~/.cache/huggingface/hub`), so nothing here needs the internet.

In [ ]:
# --- Standard library ---------------------------------------------------
import os
import re
import json
import time
from pathlib import Path

# --- Third party --------------------------------------------------------
import torch
import chromadb
import sentence_transformers
import transformers
from pypdf import PdfReader

# Print versions. When a tutorial breaks months later, it is almost always
# because one of these moved to a new major version and renamed something.
print("python              ", ".".join(map(str, __import__("sys").version_info[:3])))
print("torch               ", torch.__version__)
print("transformers        ", transformers.__version__)
print("sentence_transformers", sentence_transformers.__version__)
print("chromadb            ", chromadb.__version__)

# Force fully-offline mode: if a model is NOT already cached this will fail
# loudly instead of silently trying to download it.
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
print("\nOffline mode ON — models must come from the local cache.")

## Configuration

Every tunable value lives in this one cell. When you want to experiment
(bigger chunks? more retrieved passages?) change it **here** and re-run the
cells below — never scatter magic numbers through the code.

In [ ]:
# --- Where things live --------------------------------------------------
# Path.cwd() is wherever Jupyter was started. If you launched from the parent
# folder, fall back to the rag-ouz subfolder so the notebook still works.
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "data" / "pdfs").exists() and (PROJECT_DIR / "rag-ouz" / "data" / "pdfs").exists():
    PROJECT_DIR = PROJECT_DIR / "rag-ouz"

PDF_DIR    = PROJECT_DIR / "data" / "pdfs"     # source documents
CHROMA_DIR = PROJECT_DIR / "data" / "chroma"   # the vector database (persists on disk)
OCR_DIR    = PROJECT_DIR / "data" / "ocr"      # optional: OCR output for scanned PDFs

# --- Models -------------------------------------------------------------
# bge-m3 is multilingual (English + Thai) which matters for Thai coding manuals.
EMBED_MODEL = "BAAI/bge-m3"
# 0.5B parameters: weak on its own, fine for reading a passage and answering.
LLM_MODEL   = "Qwen/Qwen2.5-0.5B-Instruct"
DEVICE      = "cpu"

# --- Chunking -----------------------------------------------------------
CHUNK_SIZE    = 800   # characters per chunk  (~200 tokens of English)
CHUNK_OVERLAP = 120   # characters repeated between neighbours

# --- Retrieval ----------------------------------------------------------
TOP_K = 4             # how many chunks get pasted into the prompt

# --- Speed control ------------------------------------------------------
# Embedding is the slow part on a CPU. 60 pages takes ~2 minutes and covers the
# introduction, the structure of the ICD, and the mortality coding rules —
# enough for every demo question below. Set to None to index all 252 pages.
MAX_PAGES = 60

COLLECTION_NAME = "rag_ouz"

print("Project dir :", PROJECT_DIR)
print("PDFs found  :", [p.name for p in sorted(PDF_DIR.glob("*.pdf"))])
print("Page limit  :", MAX_PAGES if MAX_PAGES else "all pages")

---
# Step 1 — Get text out of the PDFs

A PDF is a **drawing format**, not a text format. It describes where to paint glyphs
on a page. Sometimes those glyphs carry their character codes (so we can recover the
text), and sometimes the page is just a **photograph of paper** (so we cannot).

`pypdf` extracts the first kind. The second kind needs OCR.

We keep the **page number** with every piece of text. That is what lets the final
answer say *"see page 118"* instead of asking you to trust it.

In [ ]:
def extract_pages(pdf_path, max_pages=None, min_chars=20):
    """Pull text out of a PDF, one page at a time.

    Returns (pages, total_page_count) where pages is a list of
    (page_number, text) tuples — 1-based page numbers, like a human reads them.

    Pages with fewer than `min_chars` characters are dropped: they are blank
    pages, or images with no text layer, and they add noise to the index.
    """
    reader = PdfReader(str(pdf_path))
    total = len(reader.pages)
    limit = total if max_pages is None else min(total, max_pages)

    pages = []
    for i in range(limit):
        # extract_text() returns None on some malformed pages, hence the "or ''"
        text = (reader.pages[i].extract_text() or "").strip()
        if len(text) >= min_chars:
            pages.append((i + 1, text))
    return pages, total


def describe(pdf_path, max_pages=None):
    """Extract, then report how much we actually got. Always look before you index."""
    t0 = time.time()
    pages, total = extract_pages(pdf_path, max_pages)
    chars = sum(len(t) for _, t in pages)
    scanned = min(total, max_pages or total)
    print(f"{pdf_path.name}")
    print(f"   pages in file      : {total}")
    print(f"   pages scanned      : {scanned}")
    print(f"   pages WITH text    : {len(pages)}")
    print(f"   characters found   : {chars:,}")
    print(f"   took               : {time.time() - t0:.1f}s")
    if len(pages) == 0:
        print("   >>> NO TEXT AT ALL — this is a scanned PDF. See the next section.")
    print()
    return pages

In [ ]:
# Run the extractor on both documents and compare the results.
documents = {}   # filename -> [(page_no, text), ...]

for pdf in sorted(PDF_DIR.glob("*.pdf")):
    documents[pdf.name] = describe(pdf, MAX_PAGES)

# Show the opening of whichever document did give us text.
for name, pages in documents.items():
    if pages:
        page_no, text = pages[0]
        print("=" * 70)
        print(f"First text page of {name} — page {page_no}")
        print("=" * 70)
        print(text[:600])
        break

---
## ⚠️ Why one of your PDFs came back empty

You just watched the two files behave completely differently:

| File | What it is | Pages | Extractable text |
|------|-----------|-------|------------------|
| **S0053.pdf** | WHO *ICD-10 Volume 2 — Instruction Manual*, 5th ed. 2016 | 252 | ~473,000 characters ✅ |
| **S0034.pdf** | *ICD-9-CM 2015* (Thai edition) | 490 | **0 characters** ❌ |

S0034 was produced by scanning paper. Every page is a single 1896×1240 CMYK JPEG.
There is no text layer — to a computer it is a picture of words, the same way a photo
of a road sign is not a string.

**This is the single most common surprise in real RAG work.** People build a pipeline,
point it at a folder of PDFs, get an empty index, and blame the embedding model.
Always measure extraction first. That is what the `describe()` cell was for.

### Your three options for S0034

1. **Skip it.** The notebook works end-to-end on S0053 alone. Recommended for learning.
2. **OCR it** — run optical character recognition to *create* the missing text layer.
   Accurate but slow, and Thai OCR needs a Thai language pack.
3. **Find a digital copy.** Always try this first. Ten minutes of searching beats
   two hours of OCR, and the quality is perfect instead of ~90%.

The next cell implements option 2, but it is **switched off by default** because
none of the required tools are installed on this machine yet.

In [ ]:
# ---------------------------------------------------------------------------
# OPTIONAL: OCR for scanned PDFs. Turn on only after installing the tools below.
# ---------------------------------------------------------------------------
RUN_OCR = False          # <-- flip to True once the install steps are done
OCR_LANG = "tha+eng"     # Thai + English; use "eng" for English-only documents
OCR_MAX_PAGES = 20       # OCR is ~2-5 seconds PER PAGE, so start tiny

# What you need to install first (none of it is present right now):
#
#   1. Tesseract OCR engine + Thai data
#      Windows: winget install --id UB-Mannheim.TesseractOCR
#      Then tick "Thai" in the installer's language list.
#
#   2. Python bindings and a PDF-to-image renderer
#      pip install pytesseract pymupdf
#
# PyMuPDF ("fitz") renders each PDF page to a bitmap; pytesseract reads it.

if RUN_OCR:
    import fitz            # PyMuPDF
    import pytesseract
    from PIL import Image

    scanned = [name for name, pages in documents.items() if not pages]
    for name in scanned:
        doc = fitz.open(PDF_DIR / name)
        out_lines = []
        limit = min(len(doc), OCR_MAX_PAGES)
        for i in range(limit):
            # Render at 200 DPI. Below ~150 accuracy collapses; above ~300 you
            # pay a lot of time for very little gain.
            pix = doc[i].get_pixmap(dpi=200)
            img = Image.frombytes("RGB", (pix.width, pix.height), pix.samples)
            text = pytesseract.image_to_string(img, lang=OCR_LANG).strip()
            if len(text) >= 20:
                out_lines.append((i + 1, text))
            print(f"  OCR {name} page {i+1}/{limit}: {len(text)} chars")

        # Cache the result — you never want to OCR the same file twice.
        OCR_DIR.mkdir(parents=True, exist_ok=True)
        cache = OCR_DIR / f"{Path(name).stem}.json"
        cache.write_text(json.dumps(out_lines, ensure_ascii=False), encoding="utf-8")
        documents[name] = out_lines
        print(f"Saved {len(out_lines)} OCR'd pages -> {cache}")
else:
    print("OCR is OFF. Continuing with the text-based PDF(s) only.")
    # Drop the empty documents so they do not pollute the rest of the pipeline.
    documents = {name: pages for name, pages in documents.items() if pages}
    print("Documents going forward:", list(documents))

---
# Step 2 — Chunking

We cannot embed a 252-page book as one vector. A single vector holds one "meaning",
and a whole book has thousands. Retrieval would become useless — every question would
match the same blob.

So we cut the text into **chunks**. The size is a genuine trade-off:

- **Too small** (say 100 chars) — a chunk says *"...shall be coded to I21.9"* with no
  indication of what "it" is. Retrieved, it is meaningless.
- **Too large** (say 5000 chars) — one chunk covers six unrelated rules. Its embedding
  is an average of all of them and matches nothing sharply. You also burn your prompt
  budget on irrelevant text.

800 characters (~200 English tokens) is a reasonable default for reference manuals.

## Why chunks overlap

Cutting at a fixed offset will eventually slice a sentence — or worse, a rule — in half:

```
chunk 1: "...the underlying cause of death is defined as"
chunk 2: "the disease which initiated the train of events..."
```

Neither chunk answers the question. By repeating the last 120 characters of each chunk
at the start of the next, any sentence near a boundary survives **intact in one of them**.
Overlap is cheap insurance against bad cuts.

In [ ]:
def clean_text(text):
    """Tidy up PDF extraction artefacts before chunking.

    PDF text arrives with hyphenation, ragged spacing and hard line breaks from
    the original layout. None of that carries meaning, and all of it wastes
    space inside a chunk.
    """
    text = text.replace("­", "")             # soft hyphens (invisible)
    text = re.sub(r"-\n(\w)", r"\1", text)        # re-join words split across lines
    text = re.sub(r"[ \t]+", " ", text)           # collapse runs of spaces
    text = re.sub(r"\n{3,}", "\n\n", text)        # at most one blank line
    return text.strip()


def chunk_text(text, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    """Split text into overlapping chunks, preferring sentence boundaries.

    Rather than cutting blindly at `size`, we look backwards for the last
    sentence end or line break in the window. If one exists in the second half
    of the window we cut there instead, so chunks end where a thought ends.
    """
    text = clean_text(text)
    if not text:
        return []
    if len(text) <= size:
        return [text]

    chunks = []
    start = 0
    while start < len(text):
        end = min(start + size, len(text))
        window = text[start:end]

        # Not the final chunk? Try to end on a natural boundary.
        if end < len(text):
            cut = max(window.rfind(". "), window.rfind("\n"))
            if cut > size * 0.5:            # only if it does not shrink us too much
                window = window[:cut + 1]
                end = start + len(window)

        piece = window.strip()
        if piece:
            chunks.append(piece)

        if end >= len(text):
            break
        start = end - overlap               # step back to create the overlap
    return chunks


# Quick self-test on a sentence we can verify by eye.
demo = ("The underlying cause of death is the disease which initiated the train of "
        "morbid events leading directly to death. " * 6)
pieces = chunk_text(demo, size=200, overlap=40)
print(f"{len(demo)} chars -> {len(pieces)} chunks of size<=200 with 40 overlap\n")
for i, p in enumerate(pieces[:3]):
    print(f"[chunk {i}] {len(p)} chars: {p[:80]}...")

In [ ]:
# Apply chunking to every page of every document, keeping the metadata that
# lets us cite a source later.
records = []   # each: {"id", "text", "source", "page"}

for name, pages in documents.items():
    for page_no, page_text in pages:
        for j, piece in enumerate(chunk_text(page_text)):
            records.append({
                "id": f"{Path(name).stem}-p{page_no}-c{j}",
                "text": piece,
                "source": name,
                "page": page_no,
            })

print(f"Total chunks: {len(records)}")
if records:
    lengths = [len(r["text"]) for r in records]
    print(f"Chunk length: min {min(lengths)}, mean {sum(lengths)//len(lengths)}, max {max(lengths)}")
    print("\n--- a sample chunk ---")
    sample = records[len(records) // 2]
    print(f"id     : {sample['id']}")
    print(f"source : {sample['source']} page {sample['page']}")
    print(f"text   : {sample['text'][:400]}...")

---
# Step 3 — Embeddings

An **embedding** is a list of numbers (1024 of them for bge-m3) that represents the
*meaning* of a piece of text. The useful property: texts that mean similar things get
vectors that point in similar directions, even with **no words in common**.

This is why RAG beats keyword search. A user asking *"how do I record what killed the
patient?"* never types "underlying cause of death" — but the two sentences land close
together in vector space.

## Measuring closeness

We compare vectors with **cosine similarity** — the cosine of the angle between them:

- `1.0` = identical direction (same meaning)
- `0.0` = perpendicular (unrelated)

We normalise every vector to length 1, which makes cosine similarity just the dot
product — faster, and it is what Chroma will do internally.

## Why bge-m3

It is **multilingual**, so an English question can retrieve Thai text and vice versa —
essential if you later OCR the Thai ICD-9-CM manual. The cost is size: 568M parameters,
about 4.3 GB on disk, and slow on a CPU. That is the trade you are making.

In [ ]:
from sentence_transformers import SentenceTransformer

print("Loading bge-m3 (~4.3 GB from local cache, takes 30-60s the first time)...")
t0 = time.time()
embedder = SentenceTransformer(EMBED_MODEL, device=DEVICE)

# bge-m3 supports 8192-token inputs, but our chunks are ~200 tokens. Capping the
# sequence length is a large CPU speed-up for zero loss in quality here.
embedder.max_seq_length = 512

print(f"Loaded in {time.time() - t0:.1f}s")
print("Embedding dimensions:", embedder.get_sentence_embedding_dimension())

In [ ]:
# Prove the claim above: meaning-similarity without shared vocabulary.
probes = [
    "How do I record what killed the patient?",                    # 0 - casual phrasing
    "Selection of the underlying cause of death for mortality.",   # 1 - manual phrasing
    "Rules for sequencing the principal diagnosis.",               # 2 - related topic
    "My neighbour is baking a chocolate cake.",                    # 3 - unrelated
]

V = embedder.encode(probes, normalize_embeddings=True)   # shape (4, 1024)
S = V @ V.T                                              # cosine similarity matrix

print("Cosine similarity\n")
print("        " + "".join(f"  [{i}]  " for i in range(len(probes))))
for i, row in enumerate(S):
    print(f"  [{i}] " + "".join(f" {v:5.2f} " for v in row))

print(f"\n[0] vs [1] = {S[0][1]:.2f}  <- different words, same meaning")
print(f"[0] vs [2] = {S[0][2]:.2f}  <- same field, different question")
print(f"[0] vs [3] = {S[0][3]:.2f}  <- unrelated")

In [ ]:
# Embed every chunk. THIS IS THE SLOW CELL — expect roughly 1-2 seconds per
# 10 chunks on a laptop CPU. With MAX_PAGES=40 that is a couple of minutes.
texts = [r["text"] for r in records]
print(f"Embedding {len(texts)} chunks on {DEVICE}...")

t0 = time.time()
vectors = embedder.encode(
    texts,
    batch_size=8,                  # small batches keep RAM use modest on CPU
    normalize_embeddings=True,     # unit length -> dot product IS cosine similarity
    show_progress_bar=True,
)
elapsed = time.time() - t0

print(f"\nDone in {elapsed:.1f}s ({elapsed / max(len(texts), 1):.2f}s per chunk)")
print("Vector matrix shape:", vectors.shape)   # (n_chunks, 1024)

---
# Step 4 — The vector store

We could keep the vectors in a NumPy array and compare the question against all of them.
With 700 chunks that works fine. With 700,000 it does not — you would scan every vector
on every query.

A **vector database** solves this with an approximate-nearest-neighbour index (Chroma
uses HNSW), which finds the closest vectors without examining them all. It also stores
the original text and metadata alongside each vector, so a search returns something you
can actually read and cite.

We use `PersistentClient`, so the index is written to `data/chroma/` on disk. Next time
you open this notebook you can skip Steps 1-3 entirely and query the existing index.

In [ ]:
# Persist to disk rather than memory, so the index survives a kernel restart.
client = chromadb.PersistentClient(path=str(CHROMA_DIR))

# Re-running this notebook should REPLACE the collection, not append to it —
# otherwise you silently end up with duplicate chunks skewing every search.
try:
    client.delete_collection(COLLECTION_NAME)
    print(f"Deleted existing collection '{COLLECTION_NAME}'")
except Exception:
    print(f"No existing collection '{COLLECTION_NAME}' (first run)")

# Chroma 1.x prefers `configuration=`; older versions used `metadata=`.
# Try the new API, fall back to the old one, so this cell works on both.
try:
    collection = client.create_collection(
        name=COLLECTION_NAME,
        configuration={"hnsw": {"space": "cosine"}},
    )
except Exception:
    collection = client.create_collection(
        name=COLLECTION_NAME,
        metadata={"hnsw:space": "cosine"},
    )

print("Collection ready:", collection.name)

In [ ]:
# Insert in batches — Chroma rejects very large single writes, and batching
# keeps memory flat.
BATCH = 200
for i in range(0, len(records), BATCH):
    part = records[i:i + BATCH]
    collection.add(
        ids       =[r["id"] for r in part],
        documents =[r["text"] for r in part],
        metadatas =[{"source": r["source"], "page": r["page"]} for r in part],
        embeddings=vectors[i:i + BATCH].tolist(),
    )
    print(f"  added {min(i + BATCH, len(records))}/{len(records)}")

print(f"\nCollection now holds {collection.count()} chunks.")
print(f"Stored on disk at: {CHROMA_DIR}")

---
# Step 5 — Retrieval

Now the payoff. To answer a question we:

1. Embed the **question** with the same model used for the chunks
   (this is mandatory — vectors from different models are not comparable),
2. ask Chroma for the `TOP_K` nearest chunk vectors,
3. get back the text and metadata.

Chroma returns **distance**, not similarity. For cosine space
`similarity = 1 - distance`, so we convert — a score near 1.0 is a strong match.

Read the scores critically. If your best hit scores 0.45, the index probably does not
contain the answer, and whatever the LLM says next is built on sand.

In [ ]:
def search(question, k=TOP_K):
    """Return the k chunks most semantically similar to `question`."""
    # Same model, same normalisation as the chunks. Non-negotiable.
    qvec = embedder.encode([question], normalize_embeddings=True)

    res = collection.query(
        query_embeddings=qvec.tolist(),
        n_results=k,
        include=["documents", "metadatas", "distances"],
    )

    # Chroma nests results one level per query; we only sent one query -> [0].
    hits = []
    for text, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
        hits.append({
            "text": text,
            "source": meta["source"],
            "page": meta["page"],
            "score": 1.0 - dist,          # cosine distance -> cosine similarity
        })
    return hits


def show_hits(question, k=TOP_K):
    """Human-readable view of what retrieval found — use this to debug bad answers."""
    print(f"QUESTION: {question}\n" + "-" * 70)
    for i, h in enumerate(search(question, k), 1):
        print(f"[{i}] score {h['score']:.3f}  |  {h['source']} p.{h['page']}")
        print(f"    {h['text'][:260].replace(chr(10), ' ')}...\n")


show_hits("What is the underlying cause of death?")

---
# Step 6 — Generation

The final step hands the retrieved chunks to a local LLM and asks it to answer
**using only those chunks**.

Qwen2.5-0.5B-Instruct is tiny — 500 million parameters, under 1 GB. On its own it
knows very little about medical coding. That is fine, and it is the whole point:
we are not asking it to *know* the answer, only to *read* one. Small models are
reasonable at extraction and summarisation even when their world knowledge is thin.

## The prompt is where grounding happens

Three instructions do the work:

1. **"Use ONLY the context"** — blocks the model from falling back on vague memory.
2. **"Cite the page"** — gives you a way to verify. An answer you cannot check is
   worth very little in a clinical setting.
3. **"If it is not in the context, say you don't know"** — gives the model an
   acceptable exit. Without it, a model under pressure will invent something.

None of these are magic. A small model will still slip. That is what Step 7 is about.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

print("Loading Qwen2.5-0.5B-Instruct (~1 GB from local cache)...")
t0 = time.time()

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
# No dtype argument: CPU wants float32, which is the default. Passing
# torch_dtype/dtype here is the #1 source of version-mismatch errors.
llm = AutoModelForCausalLM.from_pretrained(LLM_MODEL)
llm.to(DEVICE)
llm.eval()   # inference mode: disables dropout

print(f"Loaded in {time.time() - t0:.1f}s")
print("Parameters:", f"{sum(p.numel() for p in llm.parameters()) / 1e6:.0f}M")

In [ ]:
SYSTEM_PROMPT = (
    "You are a medical coding assistant. Answer the question using ONLY the "
    "context passages provided. Cite the page number for each fact you use, "
    "like [p.118]. If the context does not contain the answer, reply exactly: "
    "\"I cannot answer this from the provided documents.\" Never use outside knowledge."
)


def build_prompt(question, hits):
    """Assemble the chat prompt: system rules + context passages + question."""
    # Label every passage with its page so the model has something to cite.
    context = "\n\n".join(
        f"[p.{h['page']}] (from {h['source']})\n{h['text']}" for h in hits
    )

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Context passages:\n\n{context}\n\nQuestion: {question}"},
    ]

    # Every instruct model expects its own special-token layout. The tokenizer
    # ships the correct template, so never hand-build this string yourself.
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


def generate(prompt, max_new_tokens=300):
    """Run the model and return only the newly generated text."""
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

    with torch.no_grad():                 # no gradients needed -> less RAM, faster
        output = llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,              # greedy: same input -> same output.
                                          # Deterministic is what you want for
                                          # factual extraction. Set True (plus a
                                          # temperature) only for creative text.
            pad_token_id=tokenizer.eos_token_id,
        )

    # output contains prompt + answer; slice off the prompt tokens.
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


print("Prompt builder and generator ready.")

In [ ]:
def ask(question, k=TOP_K, show_context=False):
    """The complete RAG loop: retrieve -> build prompt -> generate -> report."""
    t0 = time.time()

    hits = search(question, k)                    # Step 5
    prompt = build_prompt(question, hits)         # Step 6a
    answer = generate(prompt)                     # Step 6b

    print("=" * 70)
    print("Q:", question)
    print("=" * 70)
    print(answer)
    print("\n" + "-" * 70)
    print("Sources retrieved:")
    for h in hits:
        print(f"   {h['source']} p.{h['page']}  (similarity {h['score']:.3f})")
    print(f"Took {time.time() - t0:.1f}s")

    if show_context:
        print("\n" + "-" * 70)
        print("EXACT CONTEXT SENT TO THE MODEL:\n")
        for h in hits:
            print(f"[p.{h['page']}] {h['text'][:300]}...\n")

    return {"question": question, "answer": answer, "hits": hits}


print("ask() is ready. Try the next cell.")

In [ ]:
# A question the manual answers directly, on page 39 of the PDF.
# Watch the retrieved page numbers — they should point near there.
_ = ask("What is the underlying cause of death and how is it defined?")

In [ ]:
# Another well-covered question (the structure of the ICD, around pages 23-25).
_ = ask("How is the ICD organised into chapters and three-character categories?")

In [ ]:
# Deliberately out of scope. A well-grounded system should REFUSE this,
# because nothing about cake is in an ICD-10 manual.
# If it answers anyway, your grounding instruction is not holding.
_ = ask("What is the best recipe for chocolate cake?")

In [ ]:
# Pass show_context=True whenever an answer looks wrong. Nine times out of ten
# the retrieval was bad, not the LLM — and this shows you immediately.
_ = ask("What is the purpose and applicability of the ICD?", show_context=True)

---
# Step 7 — Where this breaks, and what to try next

You now have a working RAG system. It is also a *naive* one. Knowing its failure
modes matters more than the code you just wrote.

## Known weaknesses of what you built

**Retrieval is the ceiling.** If Step 5 does not surface the right chunk, no LLM can
recover — it simply does not have the information. When an answer is wrong, run
`ask(..., show_context=True)` and check the retrieval **before** blaming the model.

**Pure semantic search misses exact codes.** Ask for `I21.9` and embeddings may return
passages that are *about* myocardial infarction without containing that exact string.
Production systems use **hybrid search**: semantic + keyword (BM25), scores combined.

**Fixed-size chunks cut across structure.** A table split down the middle becomes two
meaningless fragments. Structure-aware chunking (split on headings and sections) beats
character counting on well-organised manuals.

**A 0.5B model is weak.** It will occasionally ignore your instructions, cite the wrong
page, or produce a fluent but subtly wrong summary. Never put this in front of a real
coder without a human check.

**Half your corpus is unusable.** S0034 is still 490 pages of pictures.

## Exercises, roughly in order of value

1. **Set `MAX_PAGES = None`** and re-run. All 252 pages, ~700 chunks. Notice how the
   answers improve — coverage is the cheapest quality win available.
2. **Break chunking on purpose.** Set `CHUNK_SIZE = 100`, re-run Steps 2-5, and read
   the retrieved chunks. Then try `CHUNK_SIZE = 4000`. Feel both failure modes.
3. **Vary `TOP_K`.** Try 1, then 10. More context is not automatically better — a small
   model gets distracted, and you may exceed its context window.
4. **Delete the grounding sentence** from `SYSTEM_PROMPT` and re-ask the cake question.
   Watch it hallucinate. Put it back.
5. **Swap the LLM** for `Qwen/Qwen2.5-1.5B-Instruct` (needs a download). Same retrieval,
   noticeably better answers — this isolates how much the generator contributes.
6. **Add keyword search.** `pip install rank_bm25`, score chunks by BM25 as well, and
   merge the two ranked lists. This is the highest-value upgrade in the list.
7. **OCR S0034.** Follow the install notes in the OCR cell, start with
   `OCR_MAX_PAGES = 5`, and inspect the text quality before committing to 490 pages.

## Reusing the index later

The Chroma collection persists in `data/chroma/`. In a fresh session you can skip
Steps 1-4 and go straight to querying:

```python
client = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection = client.get_collection(COLLECTION_NAME)
embedder = SentenceTransformer(EMBED_MODEL, device=DEVICE)
# search() and ask() now work as before
```

## How this maps to your FastAPI project

The `../app/` backend is this same pipeline wearing a web server:

| Notebook | FastAPI project |
|----------|-----------------|
| `chunk_text()` | `app/core/chunking.py` |
| `embedder` | `app/core/embeddings.py` |
| `collection` | `app/core/vectorstore.py` |
| `generate()` | `app/core/llm.py` |
| `ask()` | `app/core/rag.py` |

Once a change works in this notebook, you know where it belongs over there.